# Mirror Counterfactual Explanations (CFE)
This notebook loads the trained ResNet-18 model and generates **Mirror CFEs** for chest X-ray predictions.

A Mirror CFE answers the question: *"What is the closest feature-space point that would flip this prediction?"*

**Pipeline:**
1. Load trained model from Drive
2. Extract feature maps from `layer4`
3. Reflect feature maps across the classifier decision boundary
4. Refine with LBFGS until the prediction flips
5. Visualise with Grad-CAM, feature difference maps, and confidence bars

## 0. Mount Drive & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import GroupShuffleSplit

torch.manual_seed(2024)
np.random.seed(2024)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_DIR   = '/content/drive/MyDrive/xray-dataset'
DATA_CSV    = os.path.join(DRIVE_DIR, 'Data_Entry_2017.csv')
MODEL_PATH  = os.path.join(DRIVE_DIR, 'resnet18_xray_final.pth')
IMAGE_GLOB  = os.path.join(DRIVE_DIR, 'images*', 'images', '*.png')

IMG_SIZE      = 224
BATCH_SIZE    = 16   # smaller batch for CFE — more memory intensive
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
CLASS_NAMES   = {0: 'No Finding', 1: 'Finding'}

print('CSV   exists:', os.path.exists(DATA_CSV))
print('Model exists:', os.path.exists(MODEL_PATH))

## 1. Load Dataset

In [ ]:
all_xray_df = pd.read_csv(DATA_CSV)
all_xray_df['Patient Age'] = all_xray_df['Patient Age'].astype(str).str.rstrip('Y').astype(int)

# Build path lookup
all_image_paths = {os.path.basename(x): x for x in glob(IMAGE_GLOB)}
all_xray_df['path'] = all_xray_df['Image Index'].map(all_image_paths.get)

# Binary label
all_xray_df['binary_label'] = (all_xray_df['Finding Labels'] != 'No Finding').astype(int)

# Drop missing paths
all_xray_df = all_xray_df.dropna(subset=['path']).copy()

print(f'Total images with valid paths: {len(all_xray_df)}')
print(all_xray_df['binary_label'].value_counts().rename({0: 'No Finding', 1: 'Finding'}))

## 2. Patient-level Train / Val / Test Split

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(splitter.split(all_xray_df, groups=all_xray_df['Patient ID']))

train_df = all_xray_df.iloc[train_idx].reset_index(drop=True)
temp_df  = all_xray_df.iloc[temp_idx].reset_index(drop=True)

splitter2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(splitter2.split(temp_df, groups=temp_df['Patient ID']))

valid_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df  = temp_df.iloc[test_idx].reset_index(drop=True)

assert len(set(train_df['Patient ID']) & set(valid_df['Patient ID'])) == 0
assert len(set(train_df['Patient ID']) & set(test_df['Patient ID']))  == 0
print('No patient overlap — split is clean')
print(f'Train: {len(train_df)}  |  Val: {len(valid_df)}  |  Test: {len(test_df)}')

## 3. Dataset & DataLoader

In [ ]:
eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class XRayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        label = torch.tensor(row['binary_label'], dtype=torch.float32)
        if self.transform:
            image = self.transform(image)
        return image, label


test_dataset = XRayDataset(test_df, transform=eval_transforms)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

print(f'Test batches: {len(test_loader)}')

## 4. Load Trained Model

In [ ]:
def build_resnet18():
    model = models.resnet18(weights=None)   # architecture only — we load our own weights
    in_features = model.fc.in_features      # 512
    model.fc = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256, 1),
        nn.Sigmoid()
    )
    return model


checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
model = build_resnet18().to(DEVICE)

# Handle both raw state_dict and wrapped checkpoint
state = checkpoint.get('model_state_dict', checkpoint)
model.load_state_dict(state)
model.eval()

print('Model loaded successfully')
print(f'Test AUC from training: {checkpoint.get("test_auc", "N/A")}')

## 5. Mirror CFE — Core Functions

### How Mirror CFE works

Given a feature vector `x0` (the feature map after `layer4`):

1. **Find the decision boundary** — defined by the classifier weights `W` and bias `b`
2. **Project** `x0` onto the boundary: `proj = x0 - ((W·x0 + b) / ||W||²) * W`
3. **Mirror** across the boundary: `mirror = 2 * proj - x0`
4. **Refine** with LBFGS so the classifier head actually outputs the target class

In [ ]:
def extract_feature_maps(model, images):
    """
    Extract feature maps from layer4 (before the FC head).
    Returns: (batch, 512, 7, 7)
    """
    feature_maps = []

    def hook_fn(module, input, output):
        feature_maps.append(output)

    hook = model.layer4.register_forward_hook(hook_fn)
    with torch.no_grad():
        _ = model(images)
    hook.remove()

    return feature_maps[0]  # (batch, 512, 7, 7)


def batch_mirror_points(x0_batch, W_batch, b_batch):
    """
    Reflect feature maps across the linear decision boundary.

    Args:
        x0_batch : (batch, 512, 7, 7)  — source feature maps
        W_batch  : (batch, 512)        — boundary normal vector per sample
        b_batch  : (batch, 1)          — boundary bias per sample

    Returns:
        mirrored  : (batch, 512, 7, 7)
        projection: (batch, 512, 7, 7)
    """
    batch_size, feature_dim, h, w = x0_batch.shape

    x0_flat = x0_batch.view(batch_size, feature_dim, -1)          # (B, 512, 49)
    W_flat  = W_batch.view(batch_size, feature_dim, 1)             # (B, 512,  1)
    b_flat  = b_batch.view(batch_size, 1, 1)                       # (B,   1,  1)

    dot_product  = torch.bmm(W_flat.transpose(1, 2), x0_flat)     # (B,  1, 49)
    norm_W_sq    = torch.sum(W_flat * W_flat, dim=1, keepdim=True) # (B,  1,  1)

    projection   = x0_flat - (dot_product + b_flat) / norm_W_sq * W_flat
    mirrored     = 2 * projection - x0_flat

    return (
        mirrored.view(batch_size, feature_dim, h, w),
        projection.view(batch_size, feature_dim, h, w)
    )


def get_mirror_points(model, mirror_point_fv, pred_logits, source_labels, cfe_labels, num_iterations=20):
    """
    Refine the initial mirror point with LBFGS so the FC head
    actually outputs the target (counterfactual) class.

    Args:
        model          : trained ResNet-18
        mirror_point_fv: initial mirror feature maps (batch, 512, 7, 7)
        pred_logits    : 2-class logits (batch, 2)
        source_labels  : original predicted class (batch,)
        cfe_labels     : target counterfactual class (batch,)
        num_iterations : LBFGS steps

    Returns:
        z: refined counterfactual feature maps (batch, 512, 7, 7)
    """
    # Swap source and target logits to define the target output
    swapped_logits = pred_logits.clone().detach()
    idx = torch.arange(cfe_labels.shape[0])
    tmp = swapped_logits[idx, cfe_labels].clone()
    swapped_logits[idx, cfe_labels]    = swapped_logits[idx, source_labels]
    swapped_logits[idx, source_labels] = tmp

    z         = Variable(mirror_point_fv.clone(), requires_grad=True)
    optimizer = torch.optim.LBFGS([z], lr=0.1)

    def closure():
        optimizer.zero_grad()
        pooled = F.adaptive_avg_pool2d(z, (1, 1))
        flat   = torch.flatten(pooled, 1)
        y      = model.fc(flat)                          # (batch, 1)
        # Expand to 2-class format to match swapped_logits
        y_2cls = torch.cat([1 - y, y], dim=1)           # (batch, 2)
        loss   = torch.norm(y_2cls - swapped_logits) ** 2
        loss.backward()
        return loss

    for _ in range(num_iterations):
        optimizer.step(closure)

    return z


def compute_mirror_cfe(model, images, device, num_iterations=20):
    """
    Full Mirror CFE pipeline for a batch of images.

    Returns:
        mirror_fv    : refined CFE feature maps (batch, 512, 7, 7)
        cfe_labels   : target labels — flipped predictions (batch,)
        source_labels: original predicted labels (batch,)
        orig_logits  : original model output probabilities (batch,)
    """
    model.eval()
    images = images.to(device)

    # 1. Original predictions
    with torch.no_grad():
        orig_logits   = model(images)                              # (batch, 1)
        source_labels = (orig_logits >= 0.5).long().squeeze(1)    # (batch,)
        cfe_labels    = 1 - source_labels                         # flip

    # 2. Feature maps from layer4
    source_fv = extract_feature_maps(model, images)               # (batch, 512, 7, 7)

    # 3. Build boundary direction from first linear layer weights
    #    model.fc[1] = Linear(512 → 256); use mean over output neurons as boundary normal
    W_mean  = model.fc[1].weight.data.mean(0)                     # (512,)
    b_mean  = model.fc[1].bias.data.mean(0, keepdim=True)         # (1,)
    batch_n = images.size(0)
    W_batch = W_mean.unsqueeze(0).expand(batch_n, -1)             # (batch, 512)
    b_batch = b_mean.unsqueeze(0).expand(batch_n, -1)             # (batch, 1)

    # 4. Initial mirror point
    mirror_fv, _ = batch_mirror_points(
        source_fv.clone().detach(),
        W_batch.detach(),
        b_batch.detach()
    )

    # 5. Refine with LBFGS
    logits_2cls = torch.cat([1 - orig_logits, orig_logits], dim=1) # (batch, 2)
    mirror_fv   = get_mirror_points(
        model, mirror_fv, logits_2cls,
        source_labels, cfe_labels,
        num_iterations=num_iterations
    )

    return mirror_fv, cfe_labels, source_labels, orig_logits.squeeze(1)


print('Mirror CFE functions defined.')

## 6. Sanity Check — Flip Rate
Run CFE on one batch and check what % of counterfactuals actually cross the decision boundary.

In [ ]:
sample_images, sample_labels = next(iter(test_loader))

mirror_fv, cfe_labels, source_labels, orig_probs = compute_mirror_cfe(
    model, sample_images, DEVICE, num_iterations=20
)

# Check how many CFEs actually flipped the prediction
with torch.no_grad():
    cfe_pooled = F.adaptive_avg_pool2d(mirror_fv, (1, 1))
    cfe_probs  = model.fc(torch.flatten(cfe_pooled, 1)).squeeze(1).cpu()
    cfe_preds  = (cfe_probs >= 0.5).long()

flip_rate = (cfe_preds == cfe_labels.cpu()).float().mean()

print(f'Batch size  : {len(sample_labels)}')
print(f'Flip rate   : {flip_rate:.2%}  (ideally > 80%)')
print(f'Source preds: {source_labels.tolist()}')
print(f'CFE preds   : {cfe_preds.tolist()}')
print(f'Target labels:{cfe_labels.tolist()}')

if flip_rate < 0.8:
    print('\nTip: flip rate is low — try increasing num_iterations to 50 in compute_mirror_cfe()')

## 7. Visualisation

Each row shows one X-ray with four panels:
- **Col 1** — Original image + Grad-CAM heatmap (where the model looked)
- **Col 2** — Feature difference map (what changed most between source and CFE)
- **Col 3** — Mean CFE feature activation + whether the prediction flipped
- **Col 4** — Confidence bars: original vs CFE probability per class

In [ ]:
def denormalise(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


def get_gradcam(model, image):
    """Grad-CAM heatmap for a single image tensor (C, H, W)."""
    target_layer = model.layer4[-1]
    gradients, activations = [], []

    h1 = target_layer.register_forward_hook(lambda m, i, o: activations.append(o))
    h2 = target_layer.register_full_backward_hook(lambda m, i, o: gradients.append(o[0]))

    model.eval()
    out = model(image.unsqueeze(0).to(DEVICE))
    model.zero_grad()
    out.backward()

    h1.remove()
    h2.remove()

    grad = gradients[0].squeeze(0)        # (512, 7, 7)
    act  = activations[0].squeeze(0)      # (512, 7, 7)
    weights = grad.mean(dim=(1, 2))       # (512,)
    cam  = F.relu((weights[:, None, None] * act).sum(dim=0))
    cam  = cam - cam.min()
    cam  = cam / (cam.max() + 1e-8)

    return cam.detach().cpu().numpy()


def visualise_mirror_cfe(model, images, labels, mirror_fv,
                          cfe_labels, source_labels, orig_probs,
                          device, n_samples=4, save_path='mirror_cfe.png'):

    model.eval()
    images = images.to(device)
    n      = min(n_samples, images.size(0))

    # CFE probabilities
    with torch.no_grad():
        cfe_pooled = F.adaptive_avg_pool2d(mirror_fv[:n], (1, 1))
        cfe_probs  = model.fc(torch.flatten(cfe_pooled, 1)).squeeze(1).cpu().numpy()

    orig_probs_np = orig_probs[:n].detach().cpu().numpy()

    # Feature difference maps
    source_fv = extract_feature_maps(model, images[:n])            # (n, 512, 7, 7)
    diff_maps = (mirror_fv[:n].detach() - source_fv).abs().mean(dim=1).cpu().numpy()  # (n, 7, 7)

    # Grad-CAM
    gradcams = [get_gradcam(model, images[i].clone().cpu()) for i in range(n)]

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, n * 4.5))
    gs  = gridspec.GridSpec(n, 4, figure=fig, hspace=0.5, wspace=0.35)

    for i in range(n):
        img_np   = denormalise(images[i].cpu()).permute(1, 2, 0).numpy()
        src_lbl  = source_labels[i].item()
        cfe_lbl  = cfe_labels[i].item()
        true_lbl = int(labels[i].item())
        flipped  = int(cfe_probs[i] >= 0.5) == cfe_lbl

        # ── Col 1: Original + Grad-CAM ───────────────────────────────────────
        ax1 = fig.add_subplot(gs[i, 0])
        ax1.imshow(img_np, cmap='bone')
        cam_resized = np.array(
            transforms.Resize((IMG_SIZE, IMG_SIZE))(
                transforms.ToPILImage()(
                    torch.tensor(gradcams[i]).unsqueeze(0).float()
                )
            )
        )
        ax1.imshow(cam_resized, cmap='jet', alpha=0.4)
        correct_colour = 'green' if src_lbl == true_lbl else 'red'
        ax1.set_title(
            f'Original + Grad-CAM\n'
            f'True: {CLASS_NAMES[true_lbl]}\n'
            f'Pred: {CLASS_NAMES[src_lbl]} ({orig_probs_np[i]:.1%})',
            fontsize=8, color=correct_colour
        )
        ax1.axis('off')

        # ── Col 2: Feature difference map ────────────────────────────────────
        ax2 = fig.add_subplot(gs[i, 1])
        diff = diff_maps[i]
        diff = (diff - diff.min()) / (diff.max() + 1e-8)
        im   = ax2.imshow(diff, cmap='hot', interpolation='bilinear')
        plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
        ax2.set_title(
            'Feature Δ (Source → CFE)\nBright = changed most\n',
            fontsize=8
        )
        ax2.axis('off')

        # ── Col 3: CFE feature map ────────────────────────────────────────────
        ax3 = fig.add_subplot(gs[i, 2])
        cfe_vis = mirror_fv[i].detach().cpu().mean(dim=0).numpy()
        cfe_vis = (cfe_vis - cfe_vis.min()) / (cfe_vis.max() + 1e-8)
        ax3.imshow(cfe_vis, cmap='viridis', interpolation='bilinear')
        flip_colour = 'green' if flipped else 'red'
        flip_text   = '✓ Prediction flipped' if flipped else '✗ Did not flip'
        ax3.set_title(
            f'CFE Feature Map\nTarget: {CLASS_NAMES[cfe_lbl]}\n{flip_text}',
            fontsize=8, color=flip_colour
        )
        ax3.axis('off')

        # ── Col 4: Confidence bars ────────────────────────────────────────────
        ax4 = fig.add_subplot(gs[i, 3])
        orig_bar = [1 - orig_probs_np[i], orig_probs_np[i]]
        cfe_bar  = [1 - cfe_probs[i],     cfe_probs[i]]
        x, width = np.arange(2), 0.35

        bars1 = ax4.bar(x - width/2, orig_bar, width, label='Original', color='steelblue', alpha=0.85)
        bars2 = ax4.bar(x + width/2, cfe_bar,  width, label='CFE',      color='tomato',    alpha=0.85)
        ax4.axhline(0.5, color='black', linestyle='--', linewidth=0.8, label='Boundary')
        ax4.set_ylim(0, 1.15)
        ax4.set_xticks(x)
        ax4.set_xticklabels(['No Finding', 'Finding'], fontsize=7)
        ax4.set_ylabel('Probability')
        ax4.set_title('Confidence\nOriginal vs CFE\n', fontsize=8)
        ax4.legend(fontsize=7)

        for bar in list(bars1) + list(bars2):
            ax4.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f'{bar.get_height():.2f}',
                ha='center', fontsize=7
            )

    plt.suptitle(
        'Mirror Counterfactual Explanations\n'
        'Col 1: Original + Grad-CAM  |  Col 2: Feature Δ  |  Col 3: CFE Features  |  Col 4: Confidence',
        fontsize=12, y=1.01
    )
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved to {save_path}')


print('Visualisation function defined.')

## 8. Run Visualisation

In [ ]:
# Use the same batch from the sanity check (already computed)
visualise_mirror_cfe(
    model         = model,
    images        = sample_images,
    labels        = sample_labels,
    mirror_fv     = mirror_fv,
    cfe_labels    = cfe_labels,
    source_labels = source_labels,
    orig_probs    = orig_probs,
    device        = DEVICE,
    n_samples     = 4,       # increase to see more rows
    save_path     = os.path.join(DRIVE_DIR, 'mirror_cfe.png')
)

## 9. Batch Evaluation — Flip Rate Across Full Test Set
Measure how reliably the Mirror CFE flips predictions across the entire test set.

In [ ]:
all_flipped, all_total = 0, 0

# Run on first 10 batches to keep it fast — remove [:10] for full test set
for batch_images, batch_labels in tqdm(list(test_loader)[:10], desc='CFE eval'):
    m_fv, c_lbls, s_lbls, o_probs = compute_mirror_cfe(
        model, batch_images, DEVICE, num_iterations=20
    )
    with torch.no_grad():
        pooled    = F.adaptive_avg_pool2d(m_fv, (1, 1))
        cfe_preds = (model.fc(torch.flatten(pooled, 1)).squeeze(1) >= 0.5).long().cpu()

    all_flipped += (cfe_preds == c_lbls.cpu()).sum().item()
    all_total   += len(batch_labels)

print(f'\nOverall flip rate: {all_flipped}/{all_total} = {all_flipped/all_total:.2%}')